In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langgraph.graph import StateGraph, END
from langgraph.types import Command
from typing import TypedDict, Annotated
import operator
from tavily import TavilyClient
import os
import re
from pydantic import BaseModel, Field
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage


In [6]:
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')

model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.3, 
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [7]:
from typing import Any, Dict, Optional, List

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    
class PlanStep(BaseModel):
    agente: str
    task: str
        
    def __repr__(self):
        return f'Agente: {self.agente}. Task: {self.task}'

class Plan(BaseModel):
    steps: List[PlanStep]

class AgentArtifact(BaseModel):
    artifact_type: str = Field(description="Il tipo di contenuto prodotto (es. 'slides', 'quiz', 'text')")
    content: Any = Field(description="Il contenuto vero e proprio (stringa, lista o dizionario)")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Note aggiuntive o formattazione")

class ToolCallerAgent:

    def __init__(self,  model, name, description, tools, system_message: str = ""):
        self.model = model.bind_tools(tools)
        self.name = name
        self.description = description
        self.tools = {t.name: t for t in tools}
        self.system_message = system_message

        graph = StateGraph(AgentState)
        graph.add_node("brain", self.brain)
        graph.add_node("action", self.action)
        graph.add_node("summarize", self.summarize)
        graph.set_entry_point("brain")

        self.graph = graph.compile()

    def brain(self, state: AgentState):
        messages = state['messages']
        if self.system_message:
            messages = [SystemMessage(content=self.system_message)] + messages
        message = self.model.invoke(messages)
        if message.tool_calls:
            
            return Command(goto='action', update={'messages': [message]})
        else:
            return Command(goto='summarize', update={'messages': [message]})

    def action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        return Command(goto='brain', update={'messages': results})

    def summarize(self, state: AgentState):
        structured_model = self.model.with_structured_output(AgentArtifact)
        
        prompt = "Riassumi i dati ottenuti dai tool in un artefatto strutturato."
        res = structured_model.invoke(state['messages'] + [HumanMessage(content=prompt)])
        print(res)
        return Command(goto=END, update = {"messages": [AIMessage(
            content=f"Ricerca completata: {res.artifact_type}",
            additional_kwargs={"artifact": res.model_dump()}
        )]})
        
        

In [8]:
class PlainAgent:

    def __init__(self,  model, name, description, system_message: str = ""):
        self.model = model
        self.name = name
        self.description = description
        self.system_message = system_message

        graph = StateGraph(AgentState)
        graph.add_node("brain", self.brain)
        graph.set_entry_point("brain")
        self.graph = graph.compile()

    def brain(self, state: AgentState):
        messages = state['messages']
        
        structured_model = self.model.with_structured_output(AgentArtifact)
        
        full_messages = [SystemMessage(content=self.system_message)] + messages
        artifact = structured_model.invoke(full_messages)
        print(artifact)
        return {'messages': [AIMessage(
            content=f"Ho generato l'artefatto di tipo: {artifact.artifact_type}",
            additional_kwargs={"artifact": artifact.model_dump()}
        )]}

In [9]:
from typing import List, Optional

class PlannerAgent:

    def __init__(self,  model, name, description,agents: List[ToolCallerAgent  | PlainAgent], system_message: str = ""):
        self.model = model
        self.name = name
        self.description = description
        self.agents = agents

        self.system_message = self._prepare_system_message(system_message)
        graph = StateGraph(AgentState)
        graph.add_node("brain", self.brain)
        graph.set_entry_point("brain")
        self.graph = graph.compile()

    def brain(self, state: AgentState):
        messages = state['messages']
        if self.system_message:
            messages = [SystemMessage(content=self.system_message)] + messages
        message = self.model.with_structured_output(Plan).invoke(messages)
        print(message)
        return {'messages': [AIMessage(
                content=f"Piano generato per l'agente: {'\n\n'.join([str(step) for step in message.steps])}",
                additional_kwargs={"plan": message.model_dump()})]}

    def _prepare_system_message(self, base_sys_message):
        agents_info = "\n".join([
            f"- {agent.name}: {agent.description}" 
            for agent in self.agents
        ])
        
        full_message = f"""{base_sys_message}
    
        ### AGENTI DISPONIBILI:
        {agents_info}
        
        ### ISTRUZIONI OPERATIVE:
        1. Usa i nomi esatti degli agenti sopra elencati.
        2. Non assegnare task ad agenti non presenti in questa lista.
        """
        return full_message

In [10]:
from typing import Any, Dict

def dict_update_reducer(left: Dict[str, Any], right: Dict[str, Any]) -> Dict[str, Any]:
    """
    'left' è lo stato attuale.
    'right' è il nuovo aggiornamento restituito da un nodo.
    """
    if left is None:
        left = {}
    if right is None:
        return left
    
    merged = left.copy()
    merged.update(right)
    return merged
    
class Task(BaseModel):
    agent: Optional[str]
    task: Optional[str]
    is_end: bool = Field(description = "Setta a True quando non ci sono ulteriori passi da fare")
    
class SupervisorState(TypedDict):
    plan: List[PlanStep]
    user_task: str
    task: str
    previous_results: Annotated[list[AnyMessage], operator.add]
    artifacts: Annotated[Dict[str, Any], dict_update_reducer]
    
    
class Supervisor:

    def __init__(self, agents: List[ToolCallerAgent | PlainAgent | PlannerAgent], system_message, model):
        self.model = model
        self.agents = agents
        self.system_message = self._prepare_system_message(system_message)
        graph = StateGraph(SupervisorState)
        graph.add_node("execute", self.execute)
        for agent in agents:
            graph.add_node(agent.name, self.create_agent_node(agent))
        graph.set_entry_point("execute")
        self.graph = graph.compile()

    def execute(self, state: SupervisorState):
        plan = state.get('plan', None)
        user_task = state.get('user_task')
        prompt = [
            SystemMessage(content=self.system_message),
            HumanMessage(content=f"User Task: {state['user_task']}. Plan: {plan}"),
        ]
        if state.get("previous_results"):
            prompt.append(HumanMessage(content=f"Previous progress: {state['previous_results']}"))
            
        task = self.model.with_structured_output(Task).invoke(prompt)
        if task.is_end:
            return Command(goto=END, update= {})
        else: 
            return Command(goto=task.agent, update={'task': task.task})
        

    def create_agent_node(self, agent):
        def agent_node(state: SupervisorState):
            task = state.get('task')
            subgraph = agent.graph
            result = subgraph.invoke({'messages': state['previous_results'] + [HumanMessage(content= f'Esegui il seguente task: {task}')]})
            last_message = result['messages'][-1]
            updates = {}
            if "artifact" in last_message.additional_kwargs:
                tipo = last_message.additional_kwargs["artifact"]["artifact_type"]
                data = last_message.additional_kwargs["artifact"]["content"]
                updates["artifacts"] = {tipo: data}
            
            if last_message.additional_kwargs.get("plan"):
                print(last_message.additional_kwargs.get("plan"))
                updates["plan"] =  Plan(**last_message.additional_kwargs.get("plan"))
                updates["previous_results"] = [last_message]
            else:
                updates["previous_results"] = [last_message]

            return Command(goto="execute", update = updates)
        return agent_node
                                    
    def _prepare_system_message(self, base_sys_message):
        agents_info = "\n".join([
            f"- {agent.name}: {agent.description}" 
            for agent in self.agents
        ])
        
        full_message = f"""{base_sys_message}
    
        ### AGENTI DISPONIBILI:
        {agents_info}
        
        ### ISTRUZIONI OPERATIVE:
        1. Usa i nomi esatti degli agenti sopra elencati.
        2. Non assegnare task ad agenti non presenti in questa lista.
        """
        return full_message       
    

In [39]:
#DEFINIAMO I TOOLS
from langchain.tools import tool
access_key = os.getenv('UNSPLASH_ACCESS_KEY')
import requests
import subprocess

@tool
def create_slides(content: List[str]):
    """
    Trasforma una lista di testi in una presentazione di slides professionale utilizzando Marp.
    
    Istruzioni per l'uso:
    1. Ogni elemento della lista 'content' deve rappresentare una slide o una porzione di essa in formato Markdown.
    2. Per inserire immagini, usa il formato speciale: $img_N(descrizione dell'immagine). 
       Esempio: "$img_0(mappa della cellula vegetale)". Il tool cercherà l'immagine e la inserirà automaticamente.
    3. Usa '---' all'interno delle stringhe per separare manualmente le slide se necessario.
    4. Il tool gestisce automaticamente il CSS per garantire che il testo sia leggibile e ben spaziato.
    
    Output: Genera un file 'output.html' visualizzabile in qualsiasi browser.
    """
    def get_unsplash_image(query: str):
        "Recupera la url di una foto da Unsplash"
        
        url = "https://api.unsplash.com/search/photos"
        
        params = {
            "query": query,
            "per_page": 1,
            "orientation": "landscape",
            "client_id": access_key
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            if data["results"]:
                # Usiamo la versione 'regular' o 'small' per non appesantire le slide
                return data["results"][0]["urls"]["regular"]
    
        # Fallback se non trova nulla
        return "https://picsum.photos/800/600"
    slides = []
    for text in content:
        image_placeholders = re.findall(r'\$img_\d+\((.*?)\)', text)
    
    # Per ogni placeholder trovato, chiamiamo l'API di Unsplash
        final_content = text
        for placeholder_text in image_placeholders:
            # Usiamo la funzione definita in precedenza
            real_url = get_unsplash_image(placeholder_text) 
            # Sostituiamo l'intero placeholder $img_N(...) con l'URL reale
            pattern = rf'\$img_\d+\({re.escape(placeholder_text)}\)'
            final_content = re.sub(pattern, '![bg right:30%](' + real_url + ')', final_content)
        slides.append(final_content)
    css = """
    section { 
        font-family: 'Arial'; 
        background-color: #fafafa; 
        font-size: 25px; /* Ridotto da 35px-40px di default */
    } 
    h1 { 
        color: #2c3e50; 
        font-size: 40px; /* Ridotto per evitare che il titolo occupi metà slide */
    }
    h2 {
        font-size: 30px;
    }
    li {
        font-size: 20px; /* Testo degli elenchi ancora più piccolo per sicurezza */
    }
    """
    full_md = f"---\nmarp: true\ntheme: default\nstyle: | \n {css}\n---\n\n" + " ".join(slides)
    
    with open("temp_slides.md", "w",encoding="utf-8") as f:
        f.write(full_md)
    subprocess.run(["npx", "@marp-team/marp-cli@latest", "temp_slides.md", "-o", "output.html"],shell=True)
    
    return {"file_path": "output.html"}
    

In [12]:
from tavily import TavilyClient

@tool
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [13]:
import wikipedia

@tool
def search_wikipedia(query: str) -> str:
    """
    Cerca su Wikipedia per approfondire concetti tecnici, nomi propri, 
    date storiche o dettagli scientifici. 
    Input: una query di ricerca specifica.
    Output: un riassunto della pagina o un elenco di suggerimenti se la query è ambigua.
    """
    wikipedia.set_lang("it")
    
    try:
        page_content = wikipedia.summary(query, sentences=5)
        return f"Risultato Wikipedia per '{query}':\n\n{page_content}"
    
    except wikipedia.exceptions.DisambiguationError as e:
        return f"La ricerca per '{query}' è ambigua. Scegli uno dei seguenti argomenti: {', '.join(e.options[:5])}"
    
    except wikipedia.exceptions.PageError:
        return f"Nessuna pagina trovata su Wikipedia per '{query}'."
    
    except Exception as e:
        return f"Errore durante la ricerca su Wikipedia: {str(e)}"

In [14]:
from google import genai
from google.genai import types
from google.genai.types import MediaResolution, ThinkingConfig, ThinkingLevel
from google.genai.types import (
    Content,
    CreateCachedContentConfig,
    FileData,
    GenerateContentConfig,
    Part,
)
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
client = genai.Client(api_key=GEMINI_API_KEY)

In [15]:
@tool
def trascribe_video_youtube(url: str):
    """Trascrivi un video youtube"""
    PROMPT = "Trascrivi il seguente video youtube"
    video_file_data = FileData(
        file_uri=LINK,
        mime_type="video/mp4",
        )
    video = Part(file_data=video_file_data)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[video, PROMPT],
    )
    return response.text

In [16]:
VIDEO_HANDLER_SYSTEM_MESSAGE = """
Sei un analista video specializzato in educazione. Il tuo compito è guardare i video YouTube forniti tramite URL e trascriverne il contenuto.

Istruzioni:
1. Usa il tool 'trascribe_video_youtube' per ottenere la trascrizione grezza del video.
2. Analizza il testo ottenuto: correggi eventuali errori di trascrizione fonetica e organizza il contenuto in paragrafi logici.
3. Restituisci un artefatto di tipo 'video_transcript' che contenga il testo pulito e un breve riassunto dei punti trattati.

Il tuo obiettivo è fornire materiale grezzo di alta qualità che il 'writer' userà per redigere la lezione."""

video_handler_agent = ToolCallerAgent(name="video_analyst", 
                                      description="Esperto di analisi video. Utilizza strumenti avanzati per trascrivere e analizzare video YouTube, estraendo concetti chiave, citazioni e spiegazioni dettagliate per alimentare la creazione della lezione.",
                                      model=model,
                                      tools=[trascribe_video_youtube],
                                      system_message=VIDEO_HANDLER_SYSTEM_MESSAGE
                                     )
                                      

In [17]:
RESEARCH_ANALYST_SYSTEM_MESSAGE = """
Sei un analista di ricerca senior specializzato nel reperimento di informazioni verificate per scopi didattici.

Il tuo obiettivo è fornire un report completo e accurato che serva da base per una lezione. Hai a disposizione due strumenti principali:
1. Wikipedia: Usalo per definizioni accademiche, contesti storici e concetti teorici stabili.
2. Tavily Search: Usalo per notizie recenti, statistiche aggiornate, esempi pratici dal mondo reale e per trovare riferimenti a immagini pertinenti.

Linee guida operative:
- Verifica incrociata: Se trovi un dato su una fonte non ufficiale, usa un altro tool per confermarlo.
- Struttura dei dati: Organizza le informazioni in sezioni chiare (es. Definizioni, Fatti Chiave, Curiosità, Esempi).
- Suggerimenti visivi: Durante la ricerca, identifica descrizioni di immagini che potrebbero essere utili per le slide (es. "Uno schema della cellula con i mitocondri in evidenza").
- Output: Una volta conclusa la ricerca, sintetizza tutto in un artefatto di tipo 'research_report'. Non limitarti a copiare e incollare, ma rielabora le informazioni per renderle utili al 'writer'.

Non fermarti alla prima risposta se il contenuto è superficiale. Sii metodico e approfondito.
"""

In [18]:
research_agent = ToolCallerAgent(name= "research_analyst", 
                                 description="Esperto di ricerca documentale e web. Utilizza Wikipedia per basi teoriche consolidate e Tavily Search per informazioni aggiornate, dati recenti e trend. Fornisce report dettagliati e verificati su qualsiasi argomento.",
                                 model = model,
                                 system_message=RESEARCH_ANALYST_SYSTEM_MESSAGE,
                                 tools = [search_wikipedia, tavily_search_tool]
                                )

In [19]:
LESSON_WRITER_SYSTEM_MESSAGE = """
Sei un Senior Educational Writer esperto in instructional design. Il tuo compito è redigere il contenuto testuale completo della lezione partendo dai materiali forniti dagli agenti di ricerca (research_analyst o video_analyst).

Il tuo obiettivo è creare un contenuto che sia pedagogicamente efficace e adatto al target di riferimento.

Istruzioni Operative:
1. Sintesi e Struttura: Organizza la lezione in moduli logici (es. Introduzione, Concetti Fondamentali, Approfondimento, Conclusione).
2. Tono di Voce: Mantieni un tono autorevole ma accessibile, incoraggiante e chiaro. Usa analogie per spiegare concetti complessi.
3. Guida Docente: Includi suggerimenti su come spiegare determinati passaggi e quali domande porre agli studenti per stimolare il dibattito.
4. Materiale di Supporto: Prepara una sintesi dei punti chiave che servirà come base per lo slide_creator.
5. Output: Restituisci un artefatto di tipo 'lesson_text'. Il 'content' deve essere un documento Markdown ben formattato, contenente sia il testo della lezione per gli studenti che le note per l'insegnante.

Non limitarti a riassumere i dati: dai loro una struttura narrativa che faciliti la memorizzazione.
"""

In [20]:
lesson_writer_agent = PlainAgent(name = "writer",
                                 description= "Esperto in divulgazione e pedagogia. Trasforma dati tecnici, trascrizioni e ricerche in una lezione fluida, chiara e coinvolgente. Si occupa di redigere il corpo del testo, le spiegazioni per il docente e le linee guida per lo studio.",
                                 model = model,
                                 system_message=LESSON_WRITER_SYSTEM_MESSAGE
)

In [21]:
SLIDER_AGENT_SYSTEM_MESSAGE = """
Sei un esperto di Visual Storytelling e Presentation Design. Il tuo compito è creare il supporto visivo per la lezione partendo dai contenuti forniti dal 'lesson_writer'.

Istruzioni Operative:
1. Sintesi Visiva: Non copiare interi paragrafi. Trasforma il testo in punti elenco (bullet points) brevi e incisivi. Ogni slide deve contenere un solo concetto chiave.
2. Struttura: Inizia sempre con una slide di titolo e termina con una slide di riepilogo. Usa il separatore '---' tra una slide e l'altra.
3. Immagini (Cruciale): Rendi la presentazione visivamente ricca. Ogni volta che un concetto può essere illustrato, inserisci un'immagine usando la sintassi: $img_N(descrizione dettagliata dell'immagine in inglese).
   - Esempio: $img_0(detailed diagram of a plant cell structure)
4. Tool: Utilizza il tool 'create_slides' passando una lista di stringhe, dove ogni stringa è il contenuto di una slide.
5. Output: Dopo aver invocato il tool, restituisci un artefatto di tipo 'presentation_slides' che confermi la creazione dell'output.html.

Regola d'oro: "Meno testo, più impatto visivo". Assicurati che le descrizioni per Unsplash siano specifiche per ottenere i risultati migliori.
"""

In [40]:
SLIDER_AGENT_SYSTEM_MESSAGE = """
Sei un esperto di Visual Storytelling e Presentation Design. Il tuo compito è creare il supporto visivo per la lezione partendo dai contenuti forniti dal 'lesson_writer'.

Istruzioni Operative:
1. Sintesi Visiva: Non copiare interi paragrafi. Trasforma il testo in punti elenco (bullet points) brevi e incisivi. Ogni slide deve contenere un solo concetto chiave.
2. Struttura: Inizia sempre con una slide di titolo e termina con una slide di riepilogo. Usa il separatore '---' tra una slide e l'altra.
3. Immagini (Cruciale): Rendi la presentazione visivamente ricca. Ogni volta che un concetto può essere illustrato, inserisci un'immagine usando la sintassi: $img_N(descrizione dettagliata dell'immagine in inglese).
   - Esempio: $img_0(detailed diagram of a plant cell structure)
4. Tool: Utilizza il tool 'create_slides' passando una lista di stringhe, dove ogni stringa è il contenuto di una slide.
5. Output: Dopo aver invocato il tool, restituisci un artefatto di tipo 'presentation_slides' che confermi la creazione dell'output.html.

Regola d'oro: "Meno testo, più impatto visivo". Assicurati che le descrizioni per Unsplash siano specifiche per ottenere i risultati migliori.
"""

In [41]:
slider_agent = ToolCallerAgent(name="slide_creator",
                               description="Designer di presentazioni didattiche. Trasforma i contenuti della lezione in slide visive efficaci, integrando immagini pertinenti e ottimizzando il layout per la comunicazione multimediale.",
                               model = model,
                               system_message = SLIDER_AGENT_SYSTEM_MESSAGE,
                               tools = [create_slides]
                              )

In [42]:
TEST_WRITER_SYSTEM_MESSAGE = """
Sei un Instructional Designer esperto nella creazione di valutazioni per l'apprendimento. 
Il tuo obiettivo è creare test che non siano banali mnemonicamente, ma che testino la reale comprensione.

REGOLE D'ORO:
1. Basati sulla 'lezione' per il perimetro dei contenuti e sul 'research_report' per la precisione dei dettagli.
2. Struttura ogni domanda con: 1 risposta corretta e 3 "distrattori" (risposte sbagliate ma plausibili).
3. Evita risposte come "Tutte le precedenti" o "Nessuna delle precedenti".
4. Se ricevi una revisione, non cambiare l'intero test se non richiesto: correggi puntualmente le criticità segnalate mantenendo lo stile e la struttura della versione precedente.
5. Il tono deve essere formale e accademico.
"""

In [43]:
test_writer_agent = PlainAgent(name="test_writer",
                               description="Esperto in valutazione e psicometria. Crea test di verifica, quiz interattivi ed esercizi di riepilogo basati sui contenuti della lezione. Progetta domande che stimolano il ragionamento critico e consolidano la memoria a lungo termine.",
                               model = model,
                               system_message=TEST_WRITER_SYSTEM_MESSAGE
)

In [44]:
PLANNER_AGENT_SYSTEM_MESSAGE = """
Sei un Senior Instructional Designer esperto nella creazione di percorsi formativi multimediali. 
Il tuo obiettivo è generare un piano d'azione (List[PlanStep]) per costruire una lezione completa basata sulla richiesta dell'utente.

Segui rigorosamente questa metodologia per definire il piano:
1. Analisi: Identifica i concetti chiave, il target e gli obiettivi di apprendimento.
2. Ricerca: Definisci task per la raccolta di dati accurati e fonti attendibili.
3. Sviluppo: Definisci task per la redazione del corpo della lezione e della guida docente.
4. Visualizzazione: Progetta la struttura delle slide, includendo istruzioni per l'inserimento di immagini tramite sintassi $img_N(descrizione).
5. Verifica: Concludi il piano con la creazione di test di valutazione.

Regole operative:
- Ogni step deve avere un obiettivo unico e chiaro.
- Assicurati che ogni task sia assegnato all'agente più pertinente tra quelli che ti verranno elencati.
- Il piano deve essere logico e sequenziale: la scrittura dipende dalla ricerca, e le slide dipendono dalla scrittura.
- Restituisci il piano esclusivamente tramite la struttura Plan fornita.
"""

In [45]:
agents = [
    video_handler_agent,
    research_agent,
    lesson_writer_agent,
    slider_agent,
    test_writer_agent
]

In [46]:
planner_agent = PlannerAgent(name = "planner", 
                             description = "Il Direttore Tecnico e Architetto del sistema. È l'unico responsabile della creazione del piano d'azione. Analizza la richiesta dell'utente, sceglie quali esperti coinvolgere e stabilisce l'ordine dei lavori per garantire una lezione coerente e completa.",
                             model = model,
                             agents = agents,
                             system_message = PLANNER_AGENT_SYSTEM_MESSAGE
                            )

In [47]:
SUPERVISOR_SYSTEM_MESSAGE = """
Sei il Supervisore Centrale del sistema Multi-Agente per la creazione di lezioni.
Il tuo compito è orchestrare il team di esperti per trasformare una richiesta utente in un pacchetto didattico completo.

Le tue Regole d'Oro:
1. Analisi dello Stato: All'inizio di ogni turno, controlla se esiste un 'plan' (Piano). Se non esiste, il tuo primo comando DEVE essere delegare al 'planner_agent'.
2. Esecuzione del Piano: Una volta ottenuto il piano, segui l'ordine dei task. Identifica il prossimo step non ancora completato analizzando i 'previous_results'.
3. Delegazione Chiara: Quando chiami un agente, trasmetti esattamente il 'task' definito nel piano. Assicurati che ogni agente sappia cosa deve fare.
4. Collezionismo di Artefatti: Ogni volta che un agente termina, verifica che abbia prodotto un artefatto (es. lesson_text, presentation_slides, final_quiz). Questi artefatti sono i mattoni della lezione finale.
5. Gestione Errori: Se un agente restituisce un errore o un contenuto palesemente incompleto, chiedigli di correggere o ripetere il task prima di procedere.
6. Conclusione: Dichiara 'is_end=True' SOLO quando tutti i punti del piano sono stati eseguiti con successo e gli artefatti necessari (testo, slide e quiz) sono stati salvati.

Obiettivo Finale: Fornire all'utente un'esperienza formativa di altissimo livello, senza lacune e pronta all'uso.
"""

In [48]:
supervisor = Supervisor(agents = agents + [planner_agent], model = model, system_message=SUPERVISOR_SYSTEM_MESSAGE)

In [49]:
#Aggiungiamo l'observability
from langfuse.langchain import CallbackHandler
import os
os.environ["LANGFUSE_SECRET_KEY"] ="sk-lf-4ad1e268-7ad3-4f0a-8967-f1deb6fe60f1"
os.environ["LANGFUSE_PUBLIC_KEY"] ="pk-lf-68f5a4fb-9cf9-4f29-a1b8-fdf0af73e2f6"
os.environ["LANGFUSE_BASE_URL"]="http://localhost:3000"
langfuse_handler = CallbackHandler()
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [50]:
result = supervisor.graph.invoke({'user_task':'Scrivimi una lezione su Dante'}, config={"callbacks": [langfuse_handler]})

steps=[Agente: research_analyst. Task: Condurre una ricerca approfondita sulla vita di Dante Alighieri, il contesto storico del Medioevo fiorentino, la struttura della Divina Commedia e l'importanza della lingua volgare., Agente: writer. Task: Redigere il corpo della lezione strutturato in: introduzione biografica, analisi del contesto politico, sintesi dell'opera principale e guida per il docente con suggerimenti pedagogici., Agente: slide_creator. Task: Progettare la presentazione visiva: Slide 1 (Titolo), Slide 2 (Biografia $img_1(ritratto di Dante)), Slide 3 (Firenze e l'esilio $img_2(mappa della Firenze medievale)), Slide 4 (Struttura della Divina Commedia $img_3(schema delle tre cantiche)), Slide 5 (Conclusione)., Agente: test_writer. Task: Creare un test di verifica composto da 5 domande a scelta multipla, 2 domande a risposta aperta sul significato allegorico dell'opera e un esercizio di associazione tra personaggi e cantiche.]
{'steps': [{'agente': 'research_analyst', 'task': 

In [156]:
result['previous_results']

[AIMessage(content='Piano generato per l\'agente: agente=\'research_analyst\' task=\'Condurre una ricerca approfondita su Dante Alighieri, focalizzandosi sulla biografia, il contesto storico-politico del Trecento, la struttura della Divina Commedia e il suo impatto sulla lingua italiana.\'\n\nagente=\'writer\' task=\'Elaborare il corpo della lezione basandosi sulla ricerca, strutturando i contenuti in moduli didattici: introduzione, vita, opere principali e analisi critica della Divina Commedia, includendo una guida per il docente.\'\n\nagente=\'slide_creator\' task="Progettare la presentazione visiva: Slide 1 (Titolo), Slide 2 (Contesto storico $img_1(mappa dell\'Italia del XIV secolo)), Slide 3 (Biografia di Dante $img_2(ritratto di Dante Alighieri)), Slide 4 (La Divina Commedia $img_3(illustrazione delle tre cantiche)), Slide 5 (Eredità culturale)."\n\nagente=\'test_writer\' task=\'Creare un test di valutazione finale composto da 10 domande a scelta multipla e 2 domande a risposta a

In [51]:
from IPython.display import IFrame
IFrame(src='./output.html', width='100%', height='500px')